# CleanVision — Production Training Notebook
### MobileNetV2 + Two-Phase Fine-Tuning + Full Evaluation

**Before running anything:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Upload your dataset to Google Drive as:
```
MyDrive/
  cleanvision_dataset/
    clean/    ← all clean floor images
    dirty/    ← all dirty floor images
```
3. Run cells **top to bottom, one by one**
4. The final output is `cleanliness_model.h5` → place in `backend/cleanliness_model.h5`

---
**Architecture:** MobileNetV2 (ImageNet pretrained) → GlobalAveragePooling → Dropout(0.3) → Dense(128, relu) → Dropout(0.2) → Dense(1, sigmoid)

**Output:** P(dirty) — matches `backend/model.py` exactly:
- `score = (1 - prediction) * 100`
- `class 0 = clean`, `class 1 = dirty` (alphabetical)

In [ ]:
#@title CELL 1 — Check GPU is active
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU is active:')
    print(result.stdout)
else:
    print('NO GPU DETECTED.')
    print('Go to Runtime → Change runtime type → T4 GPU → Save, then reconnect.')

In [ ]:
#@title CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
#@title CELL 3 — Verify dataset exists and count images
import os

DATASET_PATH = '/content/drive/MyDrive/cleanvision_dataset'  #@param {type:"string"}

print('Checking dataset at:', DATASET_PATH)
print('=' * 50)

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
class_counts = {}
total = 0

for class_name in ['clean', 'dirty']:
    class_path = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_path):
        print(f'  ERROR: {class_name}/ folder NOT FOUND at {class_path}')
        print(f'  Create it and add your images before continuing.')
        class_counts[class_name] = 0
        continue
    files = [f for f in os.listdir(class_path)
             if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS]
    class_counts[class_name] = len(files)
    total += len(files)
    print(f'  {class_name}/: {len(files)} images  OK')

print('=' * 50)
print(f'  Total: {total} images')

if total < 100:
    print('WARNING: Very few images. Add more for better accuracy.')
elif total >= 400:
    print('Dataset size: GOOD for transfer learning')

if class_counts.get('clean', 0) > 0 and class_counts.get('dirty', 0) > 0:
    ratio = class_counts['clean'] / class_counts['dirty']
    print(f'  Clean:Dirty ratio = {ratio:.1f}:1')
    if ratio > 5:
        print('  WARNING: Heavy imbalance. Class weights will compensate.')
    elif ratio < 0.5:
        print('  WARNING: More dirty than clean. Class weights will compensate.')
    else:
        print('  Ratio is acceptable.')

In [ ]:
#@title CELL 4 — Remove corrupted images (prevents training crashes)
from PIL import Image, UnidentifiedImageError
import os

removed = 0
checked = 0

for class_name in ['clean', 'dirty']:
    class_path = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_path):
        continue
    for fname in os.listdir(class_path):
        fpath = os.path.join(class_path, fname)
        if not os.path.splitext(fname)[1].lower() in VALID_EXTENSIONS:
            continue
        checked += 1
        try:
            with Image.open(fpath) as img:
                img.verify()
        except (UnidentifiedImageError, Exception):
            print(f'  Removing corrupted image: {fpath}')
            os.remove(fpath)
            removed += 1

print(f'Checked {checked} images. Removed {removed} corrupted files.')

In [ ]:
#@title CELL 5 — Imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f'TensorFlow: {tf.__version__}')
print(f'GPU devices: {tf.config.list_physical_devices("GPU")}')

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
#@title CELL 6 — Configuration (edit these if needed)

IMG_SIZE    = (224, 224)   # Must match backend/model.py exactly
BATCH_SIZE  = 16           # 16 is safer than 32 for small datasets
VAL_SPLIT   = 0.2          # 80% train, 20% validation
SEED        = 42

# Phase 1 — train head only
P1_EPOCHS   = 10
P1_LR       = 1e-4         # Low LR for stability

# Phase 2 — fine-tune last N layers of MobileNetV2
P2_EPOCHS   = 8
P2_LR       = 5e-6         # Very low LR for fine-tuning
FINETUNE_FROM = 120        # Unfreeze layers from index 100 onward

print('Config loaded.')
print(f'  Image size: {IMG_SIZE}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Phase 1: {P1_EPOCHS} epochs @ LR={P1_LR}')
print(f'  Phase 2: {P2_EPOCHS} epochs @ LR={P2_LR}, unfreeze from layer {FINETUNE_FROM}')

In [ ]:
#@title CELL 7 — Load raw datasets (NO augmentation yet)

raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=VAL_SPLIT,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=True,
)

raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False,
)

CLASS_NAMES = raw_train_ds.class_names
print(f'Class names (alphabetical order): {CLASS_NAMES}')
print()
print('IMPORTANT — this must print: ["clean", "dirty"]')
print('This means: clean=0, dirty=1 — matching backend/model.py')
print()
if CLASS_NAMES != ['clean', 'dirty']:
    print('ERROR: Class order is wrong!')
    print(f'Got: {CLASS_NAMES}')
    print('Rename your folders to exactly "clean" and "dirty" (lowercase).')
else:
    print('Class order confirmed. Backend will work correctly.')

In [ ]:
#@title CELL 8 — Calculate class weights (handles imbalance)

# Count labels efficiently from raw dataset before augmentation
all_labels = np.concatenate([
    y.numpy().flatten() for _, y in raw_train_ds
])

n_clean = int(np.sum(all_labels == 0))
n_dirty = int(np.sum(all_labels == 1))
n_total = n_clean + n_dirty

# Inverse frequency weighting
weight_clean = n_total / (2.0 * n_clean)
weight_dirty = n_total / (2.0 * n_dirty)

class_weight = {0: weight_clean, 1: weight_dirty}

print(f'Training set: {n_clean} clean, {n_dirty} dirty, {n_total} total')
print(f'class_weight: {class_weight}')
print()
print(f'Effect: dirty images count {weight_dirty/weight_clean:.1f}x more than clean')
print('This prevents the model from ignoring the minority class.')

In [ ]:
#@title CELL 9 — Preview sample images from dataset
plt.figure(figsize=(14, 6))
for images, labels in raw_train_ds.take(1):
    images_np = images.numpy().astype('uint8')
    labels_np = labels.numpy().flatten()
    n_show = min(8, len(images_np))
    for i in range(n_show):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images_np[i])
        label_name = CLASS_NAMES[int(labels_np[i])]
        color = 'green' if label_name == 'clean' else 'red'
        plt.title(label_name, color=color, fontsize=12)
        plt.axis('off')
plt.suptitle('Sample Training Images (verify labels are correct)', fontsize=13)
plt.tight_layout()
plt.show()
print('Visually verify: green=clean, red=dirty. If any label looks wrong, fix your folders.')

In [ ]:
#@title CELL 10 — Build augmentation + preprocessing pipeline

# Normalization: /255.0 — matches backend/model.py exactly
normalize = layers.Rescaling(1.0 / 255)

# Augmentation — applied to training only
# Aggressive enough to help generalization, conservative enough not to distort floor features
augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.10),          # ±10% rotation
    layers.RandomZoom(0.10),              # ±10% zoom
    layers.RandomBrightness(0.20),        # ±20% brightness
    layers.RandomContrast(0.15),          # ±15% contrast
    layers.RandomTranslation(0.05, 0.05), # small translation
], name='augmentation')

AUTOTUNE = tf.data.AUTOTUNE

# Training pipeline: normalize → augment → cache-in-RAM → prefetch
# NOTE: augment AFTER normalize so brightness/contrast work on [0,1] range
train_ds = (
    raw_train_ds
    .map(lambda x, y: (normalize(x), y), num_parallel_calls=AUTOTUNE)
    .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

# Validation: normalize only — no augmentation, no shuffling
val_ds = (
    raw_val_ds
    .map(lambda x, y: (normalize(x), y), num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)

print('Data pipeline built.')
print('Train: normalize + augment + prefetch')
print('Val:   normalize + cache + prefetch')

In [ ]:
#@title CELL 11 — Preview augmented samples (verify augmentation is sensible)
plt.figure(figsize=(14, 6))
for images, labels in train_ds.take(1):
    images_np = images.numpy()
    labels_np = labels.numpy().flatten()
    n_show = min(8, len(images_np))
    for i in range(n_show):
        plt.subplot(2, 4, i + 1)
        # Clip to [0,1] for display
        plt.imshow(np.clip(images_np[i], 0, 1))
        label_name = CLASS_NAMES[int(labels_np[i])]
        color = 'green' if label_name == 'clean' else 'red'
        plt.title(f'{label_name} (aug)', color=color, fontsize=11)
        plt.axis('off')
plt.suptitle('Augmented Training Samples', fontsize=13)
plt.tight_layout()
plt.show()
print('Augmentation looks reasonable if images are recognizable but slightly transformed.')

In [ ]:
#@title CELL 12 — Build MobileNetV2 model

# Load MobileNetV2 pretrained on ImageNet, no top layer
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze for Phase 1

print(f'MobileNetV2 loaded. Total layers: {len(base_model.layers)}')
print(f'Frozen for Phase 1 training.')

# Build classification head
# Architecture matches model.py comment exactly
inputs  = keras.Input(shape=(224, 224, 3), name='input_image')
x       = base_model(inputs, training=False)       # Pass training=False to keep BN frozen
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.3)(x)                   # Higher dropout during Phase 1
x       = layers.Dense(128, activation='relu',
                       kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x       = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid', name='cleanliness_score')(x)

model = keras.Model(inputs, outputs, name='CleanVision_MobileNetV2')

# Count trainable params
trainable   = sum([tf.size(w).numpy() for w in model.trainable_weights])
untrainable = sum([tf.size(w).numpy() for w in model.non_trainable_weights])
print(f'Trainable params (head only): {trainable:,}')
print(f'Frozen params (base):         {untrainable:,}')
print()
print('Model output: P(dirty) — score = (1 - prediction) * 100')

In [ ]:
#@title CELL 13 — PHASE 1: Train head only (base frozen)
print('=' * 60)
print('PHASE 1: Training classification head (base model frozen)')
print(f'LR={P1_LR}, max {P1_EPOCHS} epochs, early stopping patience=5')
print('=' * 60)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=P1_LR),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc'),
    ]
)

callbacks_p1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        '/content/cleanliness_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=P1_EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks_p1,
    verbose=1,
)

best_p1_acc = max(history1.history['val_accuracy'])
print()
print(f'Phase 1 complete. Best val accuracy: {best_p1_acc:.2%}')

In [ ]:
#@title CELL 14 — PHASE 2: Fine-tune last layers of MobileNetV2
print('=' * 60)
print(f'PHASE 2: Fine-tuning from layer {FINETUNE_FROM} onward')
print(f'LR={P2_LR} (10x lower than Phase 1), max {P2_EPOCHS} epochs')
print('=' * 60)

# Unfreeze the base model partially
base_model.trainable = True
for layer in base_model.layers[:FINETUNE_FROM]:
    layer.trainable = False

# Count newly trainable params
trainable_now = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f'Trainable params now: {trainable_now:,}')
print(f'Layers frozen: {FINETUNE_FROM} / {len(base_model.layers)}')
print()

# IMPORTANT: Recompile with very low LR after unfreezing
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=P2_LR),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc'),
    ]
)

callbacks_p2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        '/content/cleanliness_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-8,
        verbose=1,
    ),
]

# Continue training from where Phase 1 left off
initial_epoch = len(history1.history['loss'])

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=initial_epoch + P2_EPOCHS,
    initial_epoch=initial_epoch,
    class_weight=class_weight,
    callbacks=callbacks_p2,
    verbose=1,
)

best_p2_acc = max(history2.history['val_accuracy'])
print()
print(f'Phase 2 complete. Best val accuracy: {best_p2_acc:.2%}')
print(f'Overall improvement: {best_p1_acc:.2%} → {best_p2_acc:.2%}')

In [ ]:
#@title CELL 15 — Plot training curves (both phases)

# Merge histories
def merge_history(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

hist = merge_history(history1, history2)
p1_len = len(history1.history['loss'])
total_epochs = len(hist['loss'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(hist['loss'],     label='Train Loss',  linewidth=2)
axes[0].plot(hist['val_loss'], label='Val Loss',    linewidth=2)
axes[0].axvline(x=p1_len, color='gray', linestyle='--', label='Phase 2 start')
axes[0].set_title('Loss', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(hist['accuracy'],     label='Train Acc',  linewidth=2)
axes[1].plot(hist['val_accuracy'], label='Val Acc',    linewidth=2, color='green')
axes[1].axvline(x=p1_len, color='gray', linestyle='--', label='Phase 2 start')
axes[1].set_title('Accuracy', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(True)

# AUC
if 'auc' in hist:
    axes[2].plot(hist['auc'],     label='Train AUC', linewidth=2)
    axes[2].plot(hist['val_auc'], label='Val AUC',   linewidth=2, color='purple')
    axes[2].axvline(x=p1_len, color='gray', linestyle='--', label='Phase 2 start')
    axes[2].set_title('AUC', fontsize=13)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylim([0, 1])
    axes[2].legend()
    axes[2].grid(True)

plt.suptitle('CleanVision Training Curves (both phases)', fontsize=14)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to /content/training_curves.png')

In [ ]:
#@title CELL 16 — Full evaluation on validation set
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
!pip install scikit-learn -q

y_true = []
y_pred_raw = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_pred_raw.extend(preds.flatten().tolist())
    y_true.extend(labels.numpy().flatten().tolist())

y_true     = np.array(y_true)
y_pred_raw = np.array(y_pred_raw)
y_pred     = (y_pred_raw >= 0.5).astype(int)

# Overall accuracy
overall_acc = np.mean(y_true == y_pred)
print(f'Overall Validation Accuracy: {overall_acc:.2%}')
print()

# Per-class report
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — Val Accuracy: {overall_acc:.2%}', fontsize=13)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-class accuracy breakdown
clean_correct = np.sum((y_true == 0) & (y_pred == 0))
clean_total   = np.sum(y_true == 0)
dirty_correct = np.sum((y_true == 1) & (y_pred == 1))
dirty_total   = np.sum(y_true == 1)

print(f'Clean accuracy: {clean_correct}/{clean_total} = {clean_correct/clean_total:.2%}')
print(f'Dirty accuracy: {dirty_correct}/{dirty_total} = {dirty_correct/dirty_total:.2%}')
print()
if dirty_total > 0 and dirty_correct / dirty_total < 0.7:
    print('WARNING: Dirty recall is low (<70%). Model may miss dirty floors during demo.')
    print('Consider: lower decision threshold below (CELL 17) or add more dirty images.')
else:
    print('Dirty recall looks good for demo.')

In [ ]:
#@title CELL 17 — Find optimal decision threshold for dirty detection
# The default threshold is 0.5 (P(dirty) >= 0.5 = dirty).
# For a hospital app, missing a dirty floor is worse than a false alarm.
# This cell finds the threshold that maximises dirty recall while keeping precision reasonable.

from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_true, y_pred_raw)

# Find threshold where dirty recall >= 0.85
TARGET_RECALL = 0.85
best_threshold = 0.5
best_f1 = 0

for p, r, t in zip(precision, recall, thresholds):
    if r >= TARGET_RECALL and p > 0:
        f1 = 2 * p * r / (p + r)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = t

y_pred_tuned = (y_pred_raw >= best_threshold).astype(int)
tuned_acc    = np.mean(y_true == y_pred_tuned)

print(f'Optimal threshold for dirty recall >= {TARGET_RECALL:.0%}: {best_threshold:.3f}')
print(f'Tuned overall accuracy: {tuned_acc:.2%}')
print()
print('Classification Report (tuned threshold):')
print(classification_report(y_true, y_pred_tuned, target_names=CLASS_NAMES))
print()
print(f'If {best_threshold:.3f} is significantly different from 0.5, update backend/model.py:')
print(f'  Change line: pred_value = float(_model.predict(img_array, verbose=0)[0][0])')
print(f'  Add below:   is_dirty = pred_value >= {best_threshold:.3f}')
print(f'  OR adjust thresholds in get_status() accordingly.')

In [ ]:
#@title CELL 18 — Sanity test: predict on specific images manually
# This simulates EXACTLY what backend/model.py does.
# Test with 2-3 images you KNOW are clean and 2-3 you KNOW are dirty.

from PIL import Image
import numpy as np

def predict_image(img_path, model, threshold=0.5):
    """Matches backend/model.py _real_predict() exactly."""
    img = Image.open(img_path).convert('RGB')
    img = img.resize((224, 224), Image.LANCZOS)
    arr = np.expand_dims(np.array(img, dtype='float32') / 255.0, axis=0)
    pred = float(model.predict(arr, verbose=0)[0][0])
    score = round((1.0 - pred) * 100.0, 1)
    score = max(0.0, min(100.0, score))

    # Thresholds from backend/model.py get_status()
    if score >= 70:
        status = 'clean'
    elif score >= 40:
        status = 'needs_attention'
    else:
        status = 'dirty'

    return {'score': score, 'status': status, 'raw_pred': pred}

# Run predictions on a sample from each class
print('Running sanity predictions...')
print('=' * 55)

results = []
for class_name, expected in [('clean', 'clean'), ('dirty', 'dirty')]:
    class_path = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_path):
        continue
    files = [f for f in os.listdir(class_path)
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    # Test first 3 files from each class
    for fname in files[:3]:
        fpath = os.path.join(class_path, fname)
        result = predict_image(fpath, model)
        correct = '✅' if result['status'] != 'needs_attention' and (
            (expected == 'clean' and result['score'] >= 70) or
            (expected == 'dirty' and result['score'] < 70)
        ) else '⚠️'
        print(f'{correct} [{expected}] {fname[:30]:<30} '
              f'score={result["score"]:5.1f}  status={result["status"]}')
        results.append({'expected': expected, 'got': result['status'], 'score': result['score']})

print('=' * 55)
correct_preds = sum(1 for r in results
                    if (r['expected'] == 'clean' and r['score'] >= 70) or
                       (r['expected'] == 'dirty' and r['score'] < 70))
print(f'Sanity check: {correct_preds}/{len(results)} correct')

if correct_preds < len(results) * 0.5:
    print()
    print('ALERT: More than half of sanity predictions are wrong.')
    print('The clean/dirty class mapping may be inverted.')
    print('Check: CLASS_NAMES printed in CELL 7 — it must be ["clean", "dirty"]')
    print('If inverted, fix folders or change score formula in backend/model.py.')

In [ ]:
#@title CELL 19 — Visual prediction grid (show predictions on val images)
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
axes = axes.flatten()

sample_images = []
sample_labels = []

for images, labels in raw_val_ds.take(3):
    sample_images.extend(images.numpy())
    sample_labels.extend(labels.numpy().flatten())

sample_images = sample_images[:18]
sample_labels = sample_labels[:18]

for i, (img_np, true_label) in enumerate(zip(sample_images, sample_labels)):
    # Predict
    arr = np.expand_dims(img_np.astype('float32') / 255.0, axis=0)
    pred = float(model.predict(arr, verbose=0)[0][0])
    score = round((1.0 - pred) * 100.0, 1)
    pred_label = 0 if score >= 70 else 1  # 0=clean, 1=dirty

    axes[i].imshow(img_np.astype('uint8'))

    true_name = CLASS_NAMES[int(true_label)]
    pred_name = 'clean' if score >= 70 else 'needs_attn' if score >= 40 else 'dirty'

    correct = (true_label == 0 and score >= 70) or (true_label == 1 and score < 70)
    border_color = 'green' if correct else 'red'

    axes[i].set_title(f'True:{true_name}\nPred:{pred_name} ({score})',
                      fontsize=8,
                      color='green' if correct else 'red')
    axes[i].axis('off')
    for spine in axes[i].spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)

plt.suptitle('Validation Predictions (green=correct, red=wrong)', fontsize=13)
plt.tight_layout()
plt.savefig('/content/prediction_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grid saved to /content/prediction_grid.png')

In [ ]:
#@title CELL 20 — Final model summary and download

print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'Phase 1 best val accuracy: {best_p1_acc:.2%}')
print(f'Phase 2 best val accuracy: {best_p2_acc:.2%}')
print(f'Overall val accuracy:      {overall_acc:.2%}')
print()
print('Model saved at: /content/cleanliness_model.h5')
print()

# Verify the saved model loads and gives same output
print('Verifying saved model...')
saved_model = keras.models.load_model('/content/cleanliness_model.h5')
for images, labels in raw_val_ds.take(1):
    img_arr = images.numpy()[0:1].astype('float32') / 255.0
    orig_pred  = float(model.predict(img_arr, verbose=0)[0][0])
    saved_pred = float(saved_model.predict(img_arr, verbose=0)[0][0])
    diff = abs(orig_pred - saved_pred)
    if diff < 1e-5:
        print(f'Saved model verified — output matches (diff={diff:.2e})')
    else:
        print(f'WARNING: Saved model output differs by {diff:.4f}')

# File size
size_mb = os.path.getsize('/content/cleanliness_model.h5') / 1024 / 1024
print(f'Model file size: {size_mb:.1f} MB')
print()
print('NEXT STEPS:')
print('1. Download cleanliness_model.h5 using the cell below')
print('2. Place it at: backend/cleanliness_model.h5')
print('3. Restart Flask: python backend/app.py')
print('4. /api/health will show mock_mode: false')
print('5. Test a scan — should give real predictions')

In [ ]:
#@title CELL 21 — Download model and all charts
from google.colab import files

print('Downloading cleanliness_model.h5 ...')
files.download('/content/cleanliness_model.h5')

print('Downloading training_curves.png ...')
files.download('/content/training_curves.png')

print('Downloading confusion_matrix.png ...')
files.download('/content/confusion_matrix.png')

print('Downloading prediction_grid.png ...')
files.download('/content/prediction_grid.png')

print()
print('All downloads started.')
print()
print('Place cleanliness_model.h5 in: CleanVision/backend/cleanliness_model.h5')
print('Keep the PNG charts — use them in your project report/slides!')

## Troubleshooting

| Problem | Fix |
|---|---|
| `Class names: ['dirty', 'clean']` in CELL 7 | Your dirty folder sorts before clean — rename or move them, or swap the score formula in model.py |
| Val accuracy stuck at ~73% | All predicted as clean — class weights didn't help; try `weight_dirty = 3.0` manually |
| OOM / memory error | Reduce BATCH_SIZE to 8 in CELL 6 |
| Training very slow | Confirm GPU is selected: Runtime → Change runtime type → T4 GPU |
| `model.h5 not found` in CELL 20 | EarlyStopping triggered before any checkpoint saved; increase patience to 8 in callbacks |
| Sanity test all wrong | Class mapping inverted — in backend/model.py change `score = round((1.0 - pred_value) * 100.0, 1)` to `score = round(pred_value * 100.0, 1)` |
| Val loss bouncing wildly | Reduce P2_LR to 5e-6 in CELL 6 and retrain Phase 2 only |